In [9]:
# Your API keys / URLs live in a hidden ".env" file (never shared, never committed).
# load_dotenv() reads that file so os.getenv(...) can find those values later.
from dotenv import load_dotenv
import os

load_dotenv()  # returns True if a .env file was found and loaded

True

# 📓 1.1 — Foundational Models

This notebook covers four things, each building on the last:

1. **Model** — the AI you talk to (here it's your `qwen3-30b`).
2. **Invoke** — send it a message, get an answer back.
3. **Agent** — a model that can work in a loop and use tools, not just reply once.
4. **Streaming** — printing the answer as it's written, word by word.

Run each cell with **Shift + Enter**. Cells share state, so a variable made up top is still around further down.

## Initialising and invoking a model

Two steps here:

- **Initialise** = set up a connection to the model once, and keep it in a variable.
- **Invoke** = actually send it a message and wait for the reply.

`init_chat_model` is LangChain's universal "give me a model" helper — the same function works for OpenAI, Claude, Gemini, or (like here) your own server. The non-obvious part is the four arguments below, so they're explained in the code.

In [15]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="arc:lite",
    model_provider="openai",
    base_url=os.getenv("RADAR_OPEN_MODEL_BASE_URL"),
    api_key=os.getenv("RADAR_OPEN_MODEL_API_KEY"),
)

In [16]:
from pprint import pprint

response = model.invoke("What's the capital of the Moon?")

pprint(response)

AIMessage(content='The Moon does not have a capital because it has no government, no sovereign nations, and no permanent human inhabitants. It is a celestial body, not a country.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 34, 'prompt_tokens': 22, 'total_tokens': 56, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'arc:lite', 'system_fingerprint': 'vllm-0.23.0-406383e7', 'id': 'chatcmpl-86fbe3692341ea86', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f5b3e-1824-76c1-b27e-04608e598e86-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 22, 'output_tokens': 34, 'total_tokens': 56, 'input_token_details': {}, 'output_token_details': {}})


In [12]:
print(response.content)

The Moon does not have a capital because it has no government, no sovereign nations, and no permanent human inhabitants. It is a celestial body, not a country.


In [13]:
from pprint import pprint

pprint(response.response_metadata)

{'finish_reason': 'stop',
 'id': 'chatcmpl-86fbe3692341ea86',
 'logprobs': None,
 'model_name': 'arc:lite',
 'model_provider': 'openai',
 'system_fingerprint': 'vllm-0.23.0-406383e7',
 'token_usage': {'completion_tokens': 34,
                 'completion_tokens_details': None,
                 'prompt_tokens': 22,
                 'prompt_tokens_details': None,
                 'total_tokens': 56}}


## Customising your Model

In [14]:
model = init_chat_model(
    model="qwen3-30b",
    model_provider= "openai",
    base_url=os.getenv("RADAR_OPEN_MODEL_BASE_URL"),
    api_key=os.getenv("RADAR_OPEN_MODEL_API_KEY"),
    temperature=1.0
)

response = model.invoke("What's the capital of the Moon?")
print(response.content)

BadRequestError: Error code: 400 - {'error': {'message': "{'error': '/chat/completions: Invalid model name passed in model=qwen3-30b. Call `/v1/models` to view available models for your key.'}", 'type': 'None', 'param': 'None', 'code': '400'}}

## Model Providers

https://docs.langchain.com/oss/python/integrations/chat

In [ ]:
model = init_chat_model(
        model="qwen3-30b",
    model_provider= "openai",
    base_url=os.getenv("RADAR_OPEN_MODEL_BASE_URL"),
    api_key=os.getenv("RADAR_OPEN_MODEL_API_KEY"),

)

response = model.invoke("What's the capital of the Moon?")
print(response.content)

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

# model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")
#
# response = model.invoke("What's the capital of the Moon?")
# print(response.content)

## Initialising and invoking an agent

In [ ]:
from langchain.agents import create_agent

agent = create_agent(model=model)

In [ ]:
# agent = create_agent(model="claude-sonnet-4-5")

In [ ]:
# agent = create_agent("gpt-5-nano")

In [ ]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="What's the capital of the Moon?")]}
)

In [ ]:
from pprint import pprint

pprint(response)

In [ ]:
print(response['messages'][-1].content)

In [ ]:
from langchain.messages import AIMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="What's the capital of the Moon?"),
    AIMessage(content="The capital of the Moon is Luna City."),
    HumanMessage(content="Interesting, tell me more about Luna City")]}
)

pprint(response)

## Streaming Output

In [ ]:
i = 1
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="Tell me all about Luna City, the capital of the Moon")]},
    stream_mode="messages"
):

    # token is a message chunk with token content
    # metadata contains which node produced the token

    if i < 9:
        i += 1 # Check if there's actual content
        print(token.content, end="", flush=True)  # Print token
        print("\n")